In [ ]:
import torch
from torch import nn
from collections import defaultdict


class SmallNetwork(nn.Module):
    def __init__(self, l2i, l3i, l4i):
        super().__init__()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(3, l2i),
            nn.ReLU(),
            nn.Linear(l2i, l3i),
            nn.ReLU(),
            nn.Linear(l3i, l4i),
            nn.ReLU(),
            nn.Linear(l4i, 1),
        )

    def forward(self, x):
        return self.linear_relu_stack(x)

model = SmallNetwork(3,3,3)

# inspect weights
for name, param in model.named_parameters():
    print(name, param.shape, '\n', param.data)


linear_relu_stack.0.weight torch.Size([3, 10]) 
 tensor([[-0.0109,  0.1981,  0.2750, -0.2067,  0.2559,  0.0100, -0.1034,  0.1870,
         -0.2651,  0.1139],
        [ 0.2830,  0.2366, -0.0246, -0.0873, -0.2239,  0.3128,  0.2444, -0.0121,
          0.2336, -0.1607],
        [-0.0355, -0.2851, -0.1864, -0.0366,  0.1805,  0.2174,  0.0095,  0.0374,
          0.1153,  0.0914]])
linear_relu_stack.0.bias torch.Size([3]) 
 tensor([-0.2430, -0.0118, -0.1532])
linear_relu_stack.2.weight torch.Size([3, 3]) 
 tensor([[ 0.1527, -0.5538,  0.5244],
        [-0.2532,  0.2354,  0.0466],
        [-0.5171, -0.2230,  0.1900]])
linear_relu_stack.2.bias torch.Size([3]) 
 tensor([ 0.4030,  0.0261, -0.3475])
linear_relu_stack.4.weight torch.Size([3, 3]) 
 tensor([[-0.3311,  0.2972, -0.5448],
        [-0.4950,  0.0474,  0.4534],
        [-0.1500,  0.3842,  0.2598]])
linear_relu_stack.4.bias torch.Size([3]) 
 tensor([0.4696, 0.0696, 0.4946])
linear_relu_stack.6.weight torch.Size([1, 3]) 
 tensor([[ 0.4413, -0.

In [66]:
linear_layers = [(i, m) for i, m in enumerate(model.linear_relu_stack) if isinstance(m, nn.Linear)]

saved = [[layer.weight.data.clone(), layer.bias.data.clone()] 
         for _, layer in linear_layers]

In [67]:
len(saved)

4

In [ ]:
def remove(saved, layer, node):
    # remove row from current layer weight
    saved[layer-1][0] = torch.cat([saved[layer-1][0][:node], saved[layer-1][0][node+1:]], dim=0)
    # remove element from current layer bias
    saved[layer-1][1] = torch.cat([saved[layer-1][1][:node], saved[layer-1][1][node+1:]])
    # remove column from next layer weight
    saved[layer][0] = torch.cat([saved[layer][0][:, :node], saved[layer][0][:, node+1:]], dim=1)

def leastweight(saved):
    layer = 0
    node = 0
    v = torch.inf
    for l in range(1,len(saved)):
        importance = saved[l][0].abs().sum(dim=0)
        print(importance)
        if v >= min(importance).item():
            v = min(importance).item()
            layer = l
            node = importance.argmin().item()
    return layer, node

def removeT(saved, removals):
    by_layer = defaultdict(set)
    for layer, node in removals:
        by_layer[layer].add(node)

    for layer, nodes in by_layer.items():
        keep = [i for i in range(saved[layer - 1][0].shape[0]) if i not in nodes]
        saved[layer - 1][0] = saved[layer - 1][0][keep]
        saved[layer - 1][1] = saved[layer - 1][1][keep]
        saved[layer][0]     = saved[layer][0][:, keep]

def leastweightRank(saved):
    rankings = []
    for l in range(1, len(saved)):
        if saved[l][0].shape[1] > 2:
            importance = saved[l][0].abs().mean(dim=0)
            print(importance)
            for k, score in enumerate(importance):
                rankings.append((score.item(), l, k))
    
    rankings.sort(key=lambda x: x[0])
    return [(l, k) for _, l, k in rankings]


In [68]:
print(saved[0][0].shape[0], saved[1][0].shape[0], saved[2][0].shape[0])

3 3 3


In [ ]:
w = leastweightRank(saved)
print(w)


tensor([0.2089, 0.4048, 0.3215])
tensor([0.4055, 0.2383, 0.1191])
tensor([0.3117, 0.5179, 0.2151])
[(2, 2), (1, 0), (3, 2), (2, 1), (3, 0), (1, 2), (1, 1), (2, 0), (3, 1)]


In [58]:
w[:2]

[(2, 2), (1, 0)]

In [59]:
removeT(saved, w[:2])

In [60]:
print(saved[0][0].shape[0], saved[1][0].shape[0], saved[2][0].shape[0])

2 2 3


In [69]:
leastweight(saved)

tensor([0.3077, 0.3374, 0.2537])
tensor([0.3254, 0.2429, 0.4193])
tensor([0.4413, 0.2379, 0.2979])


[(3, 1), (2, 1), (1, 2), (3, 2), (1, 0), (2, 0), (1, 1), (2, 2), (3, 0)]

In [62]:
l2i = saved[1][0].shape[0]  # out_features of layer 0
l3i = saved[2][0].shape[0]  # out_features of layer 1
l4i = saved[3][0].shape[0]  # out_features of layer 2

new_model = SmallNetwork(l2i, l3i, l4i)
new_linear_layers = [(i, m) for i, m in enumerate(new_model.linear_relu_stack) if isinstance(m, nn.Linear)]

for (w, b), (_, layer) in zip(saved, new_linear_layers):
    layer.weight.data = w.clone()
    layer.bias.data   = b.clone()

In [63]:
for name, param in new_model.named_parameters():
    print(name, param.shape, '\n', param.data)

linear_relu_stack.0.weight torch.Size([2, 3]) 
 tensor([[ 0.1145,  0.3495,  0.0959],
        [-0.1184,  0.5706, -0.1397]])
linear_relu_stack.0.bias torch.Size([2]) 
 tensor([0.5021, 0.2796])
linear_relu_stack.2.weight torch.Size([2, 2]) 
 tensor([[-0.1448,  0.4710],
        [-0.5086,  0.4558]])
linear_relu_stack.2.bias torch.Size([2]) 
 tensor([0.4913, 0.3198])
linear_relu_stack.4.weight torch.Size([3, 2]) 
 tensor([[-0.3451, -0.2112],
        [ 0.3234,  0.0254],
        [-0.5482,  0.4783]])
linear_relu_stack.4.bias torch.Size([3]) 
 tensor([-0.3318,  0.4930,  0.4058])
linear_relu_stack.6.weight torch.Size([1, 3]) 
 tensor([[-0.3117,  0.5179, -0.2151]])
linear_relu_stack.6.bias torch.Size([1]) 
 tensor([0.1785])


In [64]:
ll = [(i, m) for i, m in enumerate(new_model.linear_relu_stack) if isinstance(m, nn.Linear)]

saved2 = [[layer.weight.data.clone(), layer.bias.data.clone()] 
         for _, layer in ll]

leastweight(saved2)

tensor([0.3117, 0.5179, 0.2151])


[(3, 2), (3, 0), (3, 1)]